<a href="https://colab.research.google.com/github/AjMing/BigData/blob/main/SparkML/Spark_ML_Regression_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing required packages
!pip install pyspark
!pip install findspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=a6d05c136601b8aa54b40b2c91fc9cadc67c9813a3bd1a3ef7ee4e501a7aad0a
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import SQLContext

In [3]:
spark = SparkSession \
    .builder \
    .appName("ML_test") \
    .getOrCreate()

# Create Spark Context

In [4]:
sc = spark.sparkContext


sqlContext = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [5]:
from google.colab import files


uploaded = files.upload()


Saving daily_weather.csv to daily_weather.csv


In [6]:
file='daily_weather.csv'

df = sqlContext.read.load(file,
                          format='com.databricks.spark.csv',
                          header='true',inferSchema='true')

#  Regression

In [7]:
from pyspark.sql import DataFrameNaFunctions
from pyspark.ml.feature import VectorAssembler , StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline


In [8]:
df = df.na.drop()

In [ ]:
featureColumns =df.columns[1:-2]

In [9]:
featureColumns =df['air_pressure_9am',
   'air_temp_9am',
 'rain_accumulation_9am',
 'relative_humidity_9am']

featureColumns=featureColumns.columns

In [10]:
df.show()

+------+-----------------+------------------+----------------------+------------------+----------------------+------------------+---------------------+-----------------+---------------------+---------------------+
|number| air_pressure_9am|      air_temp_9am|avg_wind_direction_9am|avg_wind_speed_9am|max_wind_direction_9am|max_wind_speed_9am|rain_accumulation_9am|rain_duration_9am|relative_humidity_9am|relative_humidity_3pm|
+------+-----------------+------------------+----------------------+------------------+----------------------+------------------+---------------------+-----------------+---------------------+---------------------+
|     0|918.0600000000087| 74.82200000000041|                 271.1| 2.080354199999768|    295.39999999999986| 2.863283199999908|                  0.0|              0.0|    42.42000000000046|   36.160000000000494|
|     1|917.3476881177097| 71.40384263106537|    101.93517935618371|2.4430092157340217|    140.47154847112498|3.5333236016106238|               

In [12]:
df=df.withColumnRenamed("relative_humidity_3pm","label")

assembler = VectorAssembler(inputCols=featureColumns, outputCol="features")
scaler= StandardScaler(inputCol= "features", outputCol= "scaled_features")
lr= LinearRegression(featuresCol= "scaled_features")

In [13]:
(trainingData, testData) = df.randomSplit([0.8,0.2], seed = 13234 )

In [14]:
pipeline = Pipeline(stages=[assembler,scaler, lr])

In [15]:
model = pipeline.fit(trainingData)

In [16]:
prediction = model.transform(testData)

In [18]:
prediction=prediction.select("scaled_features","label","prediction")
prediction.show()

+--------------------+------------------+------------------+
|     scaled_features|             label|        prediction|
+--------------------+------------------+------------------+
|[289.669414880693...|  68.0500000000012| 78.44809627739403|
|[290.869234644843...| 45.87000000000005|24.257598435615023|
|[291.246675205039...|14.649909361535952|23.200787693179564|
|[291.605470002849...| 17.28116117430645|16.797794264543313|
|[290.625471790385...| 47.03000000000032| 66.99518841713109|
|[289.504795810147...|  90.9900000000004|  81.3449553293434|
|[289.558613583209...| 88.35000000000014| 80.53075001805007|
|[291.872762402021...|15.849563944731992|16.574658922569938|
|[291.723987511127...| 49.50000000000024| 32.85809806312193|
|[290.584317022750...|  69.1900000000004| 64.56214713111149|
|[292.408953785502...|17.887744356437626|16.388395317939626|
|[291.947563281099...|21.987460672244843|18.549716763838774|
|[290.540813499377...|19.137323931850027|  22.9665726096581|
|[292.132369436130...| 3

In [19]:
prediction.show(truncate=False)

+------------------------------------------------------------------------------+------------------+------------------+
|scaled_features                                                               |label             |prediction        |
+------------------------------------------------------------------------------+------------------+------------------+
|[289.6694148806931,4.295888850746491,0.0,3.522952463960001]                   |68.0500000000012  |78.44809627739403 |
|[290.86923464484386,6.067135792786793,0.0,0.7515101757205789]                 |45.87000000000005 |24.257598435615023|
|[291.2466752050393,5.694723807814742,0.0,0.7479123846381086]                  |14.649909361535952|23.200787693179564|
|[291.6054700028498,6.224215649148847,0.0,0.4668986089735785]                  |17.28116117430645 |16.797794264543313|
|[290.62547179038506,4.582414091371097,0.0,3.0569366301268897]                 |47.03000000000032 |66.99518841713109 |
|[289.5047958101472,4.512410765536841,0.0,3.6621

In [ ]:
from pyspark.sql import functions as func

prediction = prediction.withColumn('label2', func.round(prediction['label'], 2))
prediction=prediction.withColumn('prediction2', func.round(prediction['prediction'], 2))

In [ ]:
prediction.select('label2','prediction2').show()

+------+-----------+
|label2|prediction2|
+------+-----------+
| 68.05|      65.58|
| 45.87|       37.4|
| 14.65|      24.87|
| 17.28|      23.03|
| 47.03|      49.85|
| 90.99|      63.44|
| 88.35|      63.74|
| 15.85|      35.18|
|  49.5|      51.18|
| 69.19|      43.13|
| 17.89|      13.25|
| 21.99|      21.95|
| 19.14|       20.9|
| 39.19|      40.76|
| 40.47|      42.23|
| 56.96|      52.56|
| 28.39|      43.81|
| 52.05|      34.89|
| 23.41|      23.94|
| 19.94|       25.3|
+------+-----------+
only showing top 20 rows



In [ ]:
#calculate rmse and r2

In [ ]:
sc.stop()